# 16 Transformer预训练与微调数据冻结

**目的**：把公共实测谱、Development有标签实测谱和15号CFM谱统一为Transformer可直接读取的峰token缓存；
为冻结的5个外折生成预训练、微调和真实谱评价索引，并预注册A–F六条CFM消融路线。

本Notebook只冻结数据，不训练模型、不读取ChemVista谱图、不查看ChemVista性能。
成功结束时状态必须为 `READY_FOR_NOTEBOOK_17`。

## 冻结的科学规则

1. 外折测试和内部验证都只包含Development公共实测谱；CFM永不进入评价索引。
2. 公共背景预训练池排除全部Development、全部ChemVista外测身份和全部CFM标签身份。
3. 同一Development化合物的CFM谱只允许进入与其自身测试折、内部验证折均不同的训练侧。
4. synthetic-only CFM同时通过14号针对外层测试折和内部验证折的Tanimoto `<0.40`权限，才可使用。
5. CFM峰数少于3的谱只保留为敏感性池，不进入主预训练或监督增强。
6. 10 eV和20 eV可进入低权重监督增强；40 eV只进入预训练。
7. 主监督权重预注册为10 eV=`0.50`、20 eV=`0.25`、40 eV=`0`；17/18号不得根据ChemVista调权。
8. 每个训练epoch仍须按化合物等概率采样，避免多谱图化合物获得更大权重。

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import pickle
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 260)
sns.set_theme(style='whitegrid', context='notebook')

# -------------------- 用户运行配置 --------------------
REBUILD_TOKEN_CACHES = False
DRY_RUN = False
# ----------------------------------------------------

# -------------------- 冻结参数 --------------------
SEED = 42
N_OUTER_FOLDS = 5
MAX_PEAKS = 120
MAX_MZ = 2000.0
MIN_PRIMARY_CFM_PEAKS = 3
CFM_SUPERVISED_WEIGHTS = {10: 0.50, 20: 0.25, 40: 0.0}
PRIMARY_CFM_PRETRAIN_ENERGIES = {10, 20, 40}
PRIMARY_CFM_SUPERVISED_ENERGIES = {10, 20}
BACKGROUND_PAIR_SEED = 42
# ------------------------------------------------

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((path for path in roots if (path / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
CFM_PLAN_DIR = BASE / 'cfm_v1' / 'frozen_generation_plan_v1'
CFM_BATCH_DIR = BASE / 'cfm_v1' / 'batch_spectra_v1'
OUTPUT_DIR = BASE / 'transformer_data_v1'
if DRY_RUN:
    OUTPUT_DIR = BASE / 'transformer_data_v1_dryrun'
CACHE_DIR = OUTPUT_DIR / 'token_caches'
FOLD_DIR = OUTPUT_DIR / 'fold_indices'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for path in (OUTPUT_DIR, CACHE_DIR, FOLD_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

PATHS = {
    'cfm15_manifest': CFM_BATCH_DIR / 'cfm_batch_run_manifest.json',
    'cfm15_qc': CFM_BATCH_DIR / 'cfm_batch_audit_qc.csv',
    'cfm_spectrum_index': CFM_BATCH_DIR / 'cfm_batch_spectrum_index.csv',
    'cfm_peaks': CFM_BATCH_DIR / 'cfm_batch_peaks.csv.gz',
    'cfm_compounds': CFM_PLAN_DIR / 'cfm_compound_manifest.csv',
    'cfm_permissions': CFM_PLAN_DIR / 'cfm_outer_fold_permission.csv',
    'cfm14_manifest': CFM_PLAN_DIR / 'cfm_generation_plan_manifest.json',
    'development': BASE / 'split_v1' / 'development_compounds.csv',
    'split_manifest': BASE / 'split_v1' / 'compound_split_manifest.csv',
    'split_freeze': BASE / 'split_v1' / 'split_freeze_manifest.json',
    'split_qc': BASE / 'split_v1' / 'leakage_checks.csv',
    'public_manifest': BASE / 'spectrum_audit_v1' / 'public_relevant_spectrum_manifest.pkl',
    'spectrum_audit': BASE / 'spectrum_audit_v1' / 'spectrum_audit_manifest.json',
    'spectrum_qc': BASE / 'spectrum_audit_v1' / 'spectrum_audit_qc_checks.csv',
    'public_stream': ROOT / '质谱数据' / 'EnvMolBench_Cleaned.pkl',
    'public_index': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Index.pkl',
    'background_pair_keys': BASE / 'set_transformer_v1' / 'background_pretraining_pair_keys.csv',
    'background_pair_cache': BASE / 'set_transformer_v1' / 'background_pretraining_pairs.npz',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少16号输入文件：\n' + '\n'.join(missing)

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.executable)
print('DRY_RUN:', DRY_RUN, 'REBUILD_TOKEN_CACHES:', REBUILD_TOKEN_CACHES)

## 1. 验证上游冻结状态与文件哈希

15号必须已经完成且建议为低权重、带域标签增强。这里仅读取冻结拆分中的外测身份排除列表，
不读取任何ChemVista谱图、峰表或预测结果。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y', 't'})

def atomic_csv(frame, path, compression=None):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig', compression=compression)
    os.replace(temporary, path)

def atomic_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

cfm15 = read_json(PATHS['cfm15_manifest'])
cfm14 = read_json(PATHS['cfm14_manifest'])
split_freeze = read_json(PATHS['split_freeze'])
spectrum_audit = read_json(PATHS['spectrum_audit'])
cfm15_qc = pd.read_csv(PATHS['cfm15_qc'])
split_qc = pd.read_csv(PATHS['split_qc'])
spectrum_qc = pd.read_csv(PATHS['spectrum_qc'])

assert cfm15['status'] == 'READY_FOR_NOTEBOOK_16'
assert cfm15['cfm_usage_recommendation'] == 'LOW_WEIGHT_AUGMENTATION_WITH_DOMAIN_LABEL'
assert cfm15['external_test_unlocked'] is False
assert as_bool(cfm15_qc['passed']).all()
assert cfm14['status'] == 'READY_FOR_NOTEBOOK_15'
assert split_freeze['qc_status'] == 'PASS' and split_freeze['n_cv_folds'] == N_OUTER_FOLDS
assert split_qc['status'].eq('PASS').all()
assert spectrum_audit['qc_status'] in {'PASS', 'PASS_WITH_WARNINGS'}
assert ~spectrum_qc['status'].eq('FAIL').any()

for filename, expected_hash in cfm15['output_hashes'].items():
    candidate = CFM_BATCH_DIR / Path(filename)
    if candidate.is_file():
        assert sha256_file(candidate) == expected_hash, f'15号输出哈希变化：{filename}'

development = pd.read_csv(PATHS['development'], low_memory=False)
split_manifest = pd.read_csv(PATHS['split_manifest'], low_memory=False)
public_manifest = pd.read_pickle(PATHS['public_manifest'])
cfm_spectra_all = pd.read_csv(PATHS['cfm_spectrum_index'])
cfm_compounds_all = pd.read_csv(PATHS['cfm_compounds'], low_memory=False)
cfm_permissions = pd.read_csv(PATHS['cfm_permissions'])

TASKS = {
    'P_screen_not_ready_binary': 'label__P_screen_not_ready_binary',
    'B_LOGBCF_REGRESSION': 'label__B_LOGBCF_REGRESSION',
    'T_EC50_MG_L_REGRESSION': 'label__T_EC50_MG_L_REGRESSION',
    'T_LC50_MG_L_REGRESSION': 'label__T_LC50_MG_L_REGRESSION',
    'T_NOEC_MG_L_REGRESSION': 'label__T_NOEC_MG_L_REGRESSION',
}
LABEL_COLUMNS = list(TASKS.values())
required_label_columns = {'inchikey_2d', 'cv_fold', *LABEL_COLUMNS}
assert required_label_columns <= set(development)

development_keys = set(development['inchikey_2d'].astype(str))
external_keys = set(split_manifest.loc[
    split_manifest['chemvista_evaluation_role'].astype(str).ne('none'), 'inchikey_2d'
].dropna().astype(str))
novel_keys = set(split_manifest.loc[
    split_manifest['chemvista_evaluation_role'].astype(str).isin(['natural_novel', 'selected_novel']), 'inchikey_2d'
].dropna().astype(str))

real_development = public_manifest.loc[as_bool(public_manifest['primary_train_eligible_20ppm'])].copy()
real_development = real_development.sort_values(['stream_offset', 'spectrum_id']).reset_index(drop=True)
included_cfm_compounds = cfm_compounds_all.loc[as_bool(cfm_compounds_all['final_include'])].copy()
valid_cfm_spectra = cfm_spectra_all.loc[
    as_bool(cfm_spectra_all['parse_success']) & as_bool(cfm_spectra_all['hard_qc_pass'])
].copy()

assert len(real_development) == int(spectrum_audit['counts']['primary_train_20ppm_spectra'])
assert real_development['inchikey_2d'].nunique() == int(spectrum_audit['counts']['primary_train_20ppm_compounds'])
assert len(valid_cfm_spectra) == int(cfm15['counts']['spectra_parsed'])
assert set(valid_cfm_spectra['inchikey_2d']) <= set(included_cfm_compounds['inchikey_2d'])
assert set(real_development['inchikey_2d']).isdisjoint(novel_keys)

if DRY_RUN:
    real_development = real_development.groupby('cv_fold', group_keys=False).head(10).reset_index(drop=True)
    valid_cfm_spectra = valid_cfm_spectra.groupby(
        ['generation_role', 'mode', 'collision_energy_ev'], group_keys=False
    ).head(10).reset_index(drop=True)

print('Development real spectra:', len(real_development), 'compounds:', real_development['inchikey_2d'].nunique())
print('Valid CFM spectra:', len(valid_cfm_spectra), 'compounds:', valid_cfm_spectra['inchikey_2d'].nunique())
print('Frozen external identities used only for exclusion:', len(external_keys))

## 2. 重建冻结的公共无标签实测谱对

沿用06号已经冻结的16,000个背景化合物，每个化合物2张谱。这里用相同随机种子从原始随机访问索引
重建两个stream offset，并逐个与既有`background_pretraining_pair_keys.csv`核对。

In [ ]:
background_pair_keys = pd.read_csv(PATHS['background_pair_keys']).sort_values('pair_index').reset_index(drop=True)
with PATHS['public_index'].open('rb') as handle:
    public_index = pickle.load(handle)

excluded_background_keys = development_keys | external_keys | set(included_cfm_compounds['inchikey_2d'].astype(str))
background_candidates = sorted(
    key for key, offsets in public_index.items()
    if key not in (development_keys | external_keys) and len(offsets) >= 2
)
rng = np.random.default_rng(BACKGROUND_PAIR_SEED)
reconstructed_keys = sorted(
    rng.choice(background_candidates, size=len(background_pair_keys), replace=False).tolist()
)
assert reconstructed_keys == background_pair_keys['inchikey_2d'].astype(str).tolist(), (
    '无法按原算法重建背景预训练身份；不要继续。'
)

background_rows = []
for pair_index, key in enumerate(reconstructed_keys):
    offsets = np.asarray(public_index[key], dtype=np.int64)
    chosen_offsets = rng.choice(offsets, size=2, replace=False)
    for side, offset in enumerate(chosen_offsets):
        background_rows.append({
            'sample_id': f'REAL_BG_{pair_index:05d}_{side}',
            'pair_index': pair_index, 'pair_side': side,
            'inchikey_2d': key, 'stream_offset': int(offset),
        })
real_background = pd.DataFrame(background_rows)
if DRY_RUN:
    real_background = real_background.loc[real_background['pair_index'] < 25].reset_index(drop=True)

assert not real_background.duplicated('sample_id').any()
assert set(real_background['inchikey_2d']).isdisjoint(development_keys)
assert set(real_background['inchikey_2d']).isdisjoint(external_keys)
assert set(real_background['inchikey_2d']).isdisjoint(set(included_cfm_compounds['inchikey_2d']))
print('Background real spectra:', len(real_background))
print('Background real compounds:', real_background['inchikey_2d'].nunique())

## 3. 统一峰token与条件编码

每个峰编码为 `[fragment_mz/2000, sqrt(relative_intensity), neutral_loss/2000, fragment/precursor]`。
谱图级连续条件包含母离子、极性、能量、能量缺失、峰数、synthetic域、稀疏谱和仪器缺失；
加合物、仪器类别和来源另存为稳定的整数编码。整数编码仅用于embedding，不具有顺序含义。

In [ ]:
ADDUCT_VOCAB = [
    'UNKNOWN', '[M+H]+', '[M+Na]+', '[M+NH4]+', '[M+K]+', '[M-H2O+H]+', '[M]+.',
    '[M-H]-', '[M+Cl]-', '[M+FA]-', '[M+Ac]-', '[M-H2O-H]-', '[M]-.', 'OTHER',
]
INSTRUMENT_VOCAB = [
    'UNKNOWN', 'ORBITRAP', 'QTOF', 'TOF', 'ION_TRAP', 'TRIPLE_QUAD', 'FTICR', 'OTHER', 'CFM_SYNTHETIC',
]
SOURCE_VOCAB = ['UNKNOWN', 'GNPS', 'MassBank', 'MoNA', 'CFM']
ADDUCT_TO_CODE = {value: index for index, value in enumerate(ADDUCT_VOCAB)}
INSTRUMENT_TO_CODE = {value: index for index, value in enumerate(INSTRUMENT_VOCAB)}
SOURCE_TO_CODE = {value: index for index, value in enumerate(SOURCE_VOCAB)}
META_NUMERIC_COLUMNS = [
    'precursor_mz_norm', 'polarity', 'collision_energy_norm', 'collision_energy_missing',
    'log_peak_count_norm', 'is_synthetic', 'sparse_lt3_peaks', 'instrument_missing',
]
PEAK_TOKEN_COLUMNS = [
    'fragment_mz_norm', 'sqrt_relative_intensity', 'neutral_loss_norm', 'fragment_precursor_ratio',
]

def instrument_category(text):
    value = str(text or '').strip().lower()
    if not value or value in {'nan', 'none', 'unknown', 'n/a'}:
        return 'UNKNOWN'
    if 'orbitrap' in value:
        return 'ORBITRAP'
    if 'qtof' in value or 'q-tof' in value or 'quadrupole time-of-flight' in value:
        return 'QTOF'
    if 'fticr' in value or 'ft-icr' in value:
        return 'FTICR'
    if 'ion trap' in value or 'iontrap' in value or 'ltq' in value:
        return 'ION_TRAP'
    if 'triple quad' in value or 'qqq' in value:
        return 'TRIPLE_QUAD'
    if 'tof' in value or 'time-of-flight' in value:
        return 'TOF'
    return 'OTHER'

def normalize_mode(text):
    value = str(text).lower()
    if 'pos' in value or value == 'positive':
        return 'positive', 1.0
    if 'neg' in value or value == 'negative':
        return 'negative', -1.0
    return 'unknown', 0.0

def encode_peaks(mz, intensity, precursor_mz):
    mz = np.asarray(mz, dtype=np.float64)
    intensity = np.asarray(intensity, dtype=np.float64)
    valid = np.isfinite(mz) & np.isfinite(intensity) & (mz > 0) & (mz < MAX_MZ) & (intensity > 0)
    mz, intensity = mz[valid], intensity[valid]
    if len(mz) > MAX_PEAKS:
        chosen = np.argsort(intensity, kind='stable')[-MAX_PEAKS:]
        mz, intensity = mz[chosen], intensity[chosen]
    if len(mz):
        order = np.argsort(mz, kind='stable')
        mz, intensity = mz[order], intensity[order]
        intensity = intensity / intensity.max()
    token = np.zeros((MAX_PEAKS, len(PEAK_TOKEN_COLUMNS)), dtype=np.float16)
    padding = np.ones(MAX_PEAKS, dtype=bool)
    if len(mz):
        precursor = float(precursor_mz) if np.isfinite(precursor_mz) and precursor_mz > 0 else np.nan
        neutral_loss = np.maximum(precursor - mz, 0.0) if np.isfinite(precursor) else np.zeros(len(mz))
        ratio = mz / precursor if np.isfinite(precursor) else np.zeros(len(mz))
        token[:len(mz), 0] = np.clip(mz / MAX_MZ, 0, 1).astype(np.float16)
        token[:len(mz), 1] = np.sqrt(intensity).astype(np.float16)
        token[:len(mz), 2] = np.clip(neutral_loss / MAX_MZ, 0, 1).astype(np.float16)
        token[:len(mz), 3] = np.clip(ratio, 0, 1.5).astype(np.float16)
        padding[:len(mz)] = False
    return token, padding, len(mz)

def token_hash(token, padding):
    valid = np.asarray(token)[~np.asarray(padding)]
    return hashlib.sha256(valid.astype('<f2').tobytes()).hexdigest()

def metadata_vector(precursor, polarity, ce, peak_n, is_synthetic, sparse, instrument_missing):
    ce_missing = not np.isfinite(ce) or ce < 0
    precursor_missing = not np.isfinite(precursor) or precursor <= 0
    return np.asarray([
        0.0 if precursor_missing else np.clip(precursor / MAX_MZ, 0, 1.5),
        polarity,
        0.0 if ce_missing else np.clip(ce / 100.0, 0, 2.0),
        float(ce_missing),
        math.log1p(peak_n) / math.log1p(MAX_PEAKS),
        float(is_synthetic), float(sparse), float(instrument_missing),
    ], dtype=np.float32)

def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)

vocab_payload = {
    'adduct': ADDUCT_VOCAB,
    'instrument_category': INSTRUMENT_VOCAB,
    'source_library': SOURCE_VOCAB,
    'unknown_policy': 'map unseen category to UNKNOWN/OTHER; never infer from endpoint label',
}
atomic_json(vocab_payload, OUTPUT_DIR / 'transformer_categorical_vocab.json')
atomic_json({
    'max_peaks': MAX_PEAKS, 'max_mz': MAX_MZ,
    'peak_token_columns': PEAK_TOKEN_COLUMNS,
    'metadata_numeric_columns': META_NUMERIC_COLUMNS,
    'peak_order': 'ascending fragment m/z after top-intensity truncation',
    'intensity_transform': 'sqrt(max-normalized intensity)',
    'missingness': 'explicit indicators; missing collision energy encoded value 0 plus missing=1',
}, OUTPUT_DIR / 'transformer_token_schema.json')

## 4. Development实测谱token缓存

只使用03号冻结的20 ppm主训练清单。逐谱图通过stream offset回读原始清洗记录并验证二维InChIKey；
不从CSV重建峰，从而保留已冻结的峰级清洗结果。

In [ ]:
def build_real_token_cache(source_index, cache_path, index_path, pool_name):
    if cache_path.exists() and index_path.exists() and not REBUILD_TOKEN_CACHES:
        frozen_index = pd.read_csv(index_path)
        with np.load(cache_path) as cached:
            assert cached['peak_tokens'].shape[0] == len(frozen_index)
        print('Reuse:', cache_path.name, len(frozen_index))
        return frozen_index

    frame = source_index.copy().reset_index(drop=True)
    frame['cache_row'] = np.arange(len(frame), dtype=int)
    features = np.zeros((len(frame), MAX_PEAKS, len(PEAK_TOKEN_COLUMNS)), dtype=np.float16)
    padding = np.ones((len(frame), MAX_PEAKS), dtype=bool)
    metadata = np.zeros((len(frame), len(META_NUMERIC_COLUMNS)), dtype=np.float32)
    adduct_code = np.zeros(len(frame), dtype=np.int16)
    instrument_code = np.zeros(len(frame), dtype=np.int16)
    source_code = np.zeros(len(frame), dtype=np.int16)
    detail_rows = []

    requests = frame.sort_values('stream_offset').itertuples(index=False)
    with PATHS['public_stream'].open('rb') as stream:
        for row in tqdm(requests, total=len(frame), desc=f'Build {pool_name} real tokens'):
            stream.seek(int(row.stream_offset))
            actual_key, record = pickle.load(stream)
            assert str(actual_key) == str(row.inchikey_2d)
            assert str(record.get('source_library')) != 'ChemVista'
            raw_tensor = np.asarray(record['tensor'], dtype=np.float32)
            raw_padding = np.asarray(record['padding_mask'], dtype=bool)
            valid = ~raw_padding
            valid[0] = False
            mz = raw_tensor[valid, 0].astype(np.float64) * MAX_MZ
            intensity = raw_tensor[valid, 3].astype(np.float64)
            precursor = float(record.get('precursor_mz', np.nan))
            token, pad, retained_n = encode_peaks(mz, intensity, precursor)
            mode, polarity = normalize_mode(record.get('ionmode', ''))
            try:
                ce = float(record.get('ce_real', np.nan))
            except (TypeError, ValueError):
                ce = np.nan
            inst = instrument_category(record.get('instrument', ''))
            adduct = str(record.get('adduct') or 'UNKNOWN')
            source = str(record.get('source_library') or 'UNKNOWN')
            cache_row = int(row.cache_row)
            features[cache_row] = token
            padding[cache_row] = pad
            metadata[cache_row] = metadata_vector(
                precursor, polarity, ce, retained_n, False, retained_n < MIN_PRIMARY_CFM_PEAKS, inst == 'UNKNOWN'
            )
            adduct_code[cache_row] = ADDUCT_TO_CODE.get(adduct, ADDUCT_TO_CODE['OTHER'])
            instrument_code[cache_row] = INSTRUMENT_TO_CODE[inst]
            source_code[cache_row] = SOURCE_TO_CODE.get(source, SOURCE_TO_CODE['UNKNOWN'])
            spectrum_id = getattr(row, 'spectrum_id', f'{pool_name}:{cache_row}')
            detail_rows.append({
                'cache_row': cache_row, 'sample_id': str(getattr(row, 'sample_id', spectrum_id)),
                'spectrum_id': str(spectrum_id), 'inchikey_2d': str(actual_key),
                'domain': 'real', 'source_pool': pool_name, 'source_library': source,
                'stream_offset': int(row.stream_offset), 'precursor_mz': precursor,
                'mode': mode, 'adduct': adduct, 'collision_energy_value': ce,
                'collision_energy_missing': not np.isfinite(ce) or ce < 0,
                'instrument_category': inst, 'retained_peak_n': retained_n,
                'sparse_lt3_peaks': retained_n < MIN_PRIMARY_CFM_PEAKS,
                'token_hash': token_hash(token, pad),
                'spectrum_hash': str(record.get('spectrum_hash', '')),
                'pair_index': getattr(row, 'pair_index', np.nan),
                'pair_side': getattr(row, 'pair_side', np.nan),
                'cv_group': getattr(row, 'cv_group', ''),
                'cv_fold': getattr(row, 'cv_fold', np.nan),
            })

    frozen_index = pd.DataFrame(detail_rows).sort_values('cache_row').reset_index(drop=True)
    atomic_npz(
        cache_path, peak_tokens=features, padding_mask=padding, metadata_numeric=metadata,
        adduct_code=adduct_code, instrument_code=instrument_code, source_code=source_code,
    )
    atomic_csv(frozen_index, index_path, compression='gzip')
    return frozen_index

dev_source = real_development.copy()
dev_source['sample_id'] = dev_source['spectrum_id'].astype(str)
real_dev_index = build_real_token_cache(
    dev_source,
    CACHE_DIR / 'real_development_tokens.npz',
    OUTPUT_DIR / 'real_development_spectrum_index.csv.gz',
    'development_labeled_real',
)
assert len(real_dev_index) == len(real_development)
display(real_dev_index.head(3))

## 5. 公共背景实测谱token缓存

背景池固定为16,000个身份、每个身份2张实测谱；它不含Development、ChemVista外测或CFM标签身份，
因而可在全部外折用于无标签实测谱预训练。

In [ ]:
real_bg_index = build_real_token_cache(
    real_background,
    CACHE_DIR / 'real_background_tokens.npz',
    OUTPUT_DIR / 'real_background_pretraining_index.csv.gz',
    'background_unlabeled_real',
)
assert len(real_bg_index) == len(real_background)
assert real_bg_index['inchikey_2d'].nunique() * 2 == len(real_bg_index)
assert set(real_bg_index['inchikey_2d']).isdisjoint(development_keys | external_keys)
display(real_bg_index.head(3))

## 6. CFM峰token缓存与质量角色

全部17,346张成功且通过硬物理QC的谱均进入缓存，但是否能用于主预训练/监督由独立字段控制。
这样保留失败与稀疏性的完整证据，同时不让低信息谱污染主分析。

In [ ]:
CFM_CACHE_PATH = CACHE_DIR / 'cfm_tokens.npz'
CFM_INDEX_PATH = OUTPUT_DIR / 'cfm_transformer_spectrum_index.csv.gz'

if CFM_CACHE_PATH.exists() and CFM_INDEX_PATH.exists() and not REBUILD_TOKEN_CACHES:
    cfm_index = pd.read_csv(CFM_INDEX_PATH)
    with np.load(CFM_CACHE_PATH) as cached:
        assert cached['peak_tokens'].shape[0] == len(cfm_index)
    print('Reuse:', CFM_CACHE_PATH.name, len(cfm_index))
else:
    required_task_ids = set(valid_cfm_spectra['spectrum_task_id'].astype(str))
    cfm_peaks = pd.read_csv(PATHS['cfm_peaks'])
    cfm_peaks = cfm_peaks.loc[cfm_peaks['spectrum_task_id'].astype(str).isin(required_task_ids)].copy()
    peak_groups = {key: group for key, group in cfm_peaks.groupby('spectrum_task_id', sort=False)}
    assert required_task_ids == set(peak_groups)

    cfm_frame = valid_cfm_spectra.sort_values('spectrum_task_id').reset_index(drop=True)
    cfm_frame['cache_row'] = np.arange(len(cfm_frame), dtype=int)
    features = np.zeros((len(cfm_frame), MAX_PEAKS, len(PEAK_TOKEN_COLUMNS)), dtype=np.float16)
    padding = np.ones((len(cfm_frame), MAX_PEAKS), dtype=bool)
    metadata = np.zeros((len(cfm_frame), len(META_NUMERIC_COLUMNS)), dtype=np.float32)
    adduct_code = np.zeros(len(cfm_frame), dtype=np.int16)
    instrument_code = np.full(len(cfm_frame), INSTRUMENT_TO_CODE['CFM_SYNTHETIC'], dtype=np.int16)
    source_code = np.full(len(cfm_frame), SOURCE_TO_CODE['CFM'], dtype=np.int16)
    detail_rows = []

    for row in tqdm(cfm_frame.itertuples(index=False), total=len(cfm_frame), desc='Build CFM tokens'):
        peaks = peak_groups[str(row.spectrum_task_id)]
        token, pad, retained_n = encode_peaks(
            peaks['fragment_mz'].to_numpy(float), peaks['relative_intensity'].to_numpy(float),
            float(row.theoretical_precursor_mz),
        )
        mode, polarity = normalize_mode(row.mode)
        energy = int(row.collision_energy_ev)
        sparse = retained_n < MIN_PRIMARY_CFM_PEAKS
        cache_row = int(row.cache_row)
        features[cache_row] = token
        padding[cache_row] = pad
        metadata[cache_row] = metadata_vector(
            float(row.theoretical_precursor_mz), polarity, float(energy), retained_n,
            True, sparse, True,
        )
        adduct_code[cache_row] = ADDUCT_TO_CODE.get(str(row.adduct), ADDUCT_TO_CODE['OTHER'])
        primary_pretrain = (not sparse) and energy in PRIMARY_CFM_PRETRAIN_ENERGIES
        primary_supervised = (not sparse) and energy in PRIMARY_CFM_SUPERVISED_ENERGIES
        if energy == 10:
            quality_tier = 'high_similarity_sparse_risk'
        elif energy == 20:
            quality_tier = 'moderate_similarity_low_weight'
        else:
            quality_tier = 'low_similarity_pretraining_only'
        detail_rows.append({
            'cache_row': cache_row, 'sample_id': str(row.spectrum_task_id),
            'spectrum_task_id': str(row.spectrum_task_id), 'job_id': str(row.job_id),
            'cfm_compound_id': str(row.cfm_compound_id), 'inchikey_2d': str(row.inchikey_2d),
            'domain': 'synthetic', 'source_pool': 'cfm', 'generation_role': str(row.generation_role),
            'mode': mode, 'adduct': str(row.adduct), 'collision_energy_value': energy,
            'precursor_mz': float(row.theoretical_precursor_mz), 'original_peak_n': int(row.peak_n),
            'retained_peak_n': retained_n, 'sparse_lt3_peaks': sparse,
            'quality_tier': quality_tier,
            'primary_pretraining_eligible': primary_pretrain,
            'sensitivity_pretraining_eligible': True,
            'primary_supervised_eligible': primary_supervised,
            'primary_supervised_sample_weight': CFM_SUPERVISED_WEIGHTS[energy],
            'token_hash': token_hash(token, pad),
        })

    cfm_index = pd.DataFrame(detail_rows).sort_values('cache_row').reset_index(drop=True)
    atomic_npz(
        CFM_CACHE_PATH, peak_tokens=features, padding_mask=padding, metadata_numeric=metadata,
        adduct_code=adduct_code, instrument_code=instrument_code, source_code=source_code,
    )
    atomic_csv(cfm_index, CFM_INDEX_PATH, compression='gzip')

cfm_compound_columns = [
    'inchikey_2d', 'generation_role', 'supervised_compound_role', 'cv_group', 'cv_fold',
    'max_tanimoto_to_chemvista_novel', *LABEL_COLUMNS,
]
cfm_index = cfm_index.merge(
    included_cfm_compounds[cfm_compound_columns], on=['inchikey_2d', 'generation_role'],
    how='left', validate='many_to_one',
)
assert cfm_index[LABEL_COLUMNS].notna().any(axis=1).all()
display(cfm_index.groupby(['generation_role', 'mode', 'collision_energy_value']).agg(
    spectra=('sample_id', 'size'), sparse=('sparse_lt3_peaks', 'sum'),
    primary_pretrain=('primary_pretraining_eligible', 'sum'),
    primary_supervised=('primary_supervised_eligible', 'sum'),
).reset_index())

## 7. 冻结统一化合物标签表

标签始终在化合物层保存。谱图行只引用二维InChIKey；18号使用masked multi-task loss，缺失标签不填零。
synthetic-only化合物只来自14号冻结表。

In [ ]:
dev_labels = development[['inchikey_2d', 'cv_group', 'cv_fold', *LABEL_COLUMNS]].copy()
dev_labels['compound_data_role'] = 'development'
synthetic_labels = included_cfm_compounds.loc[
    included_cfm_compounds['generation_role'].eq('synthetic_only_new_labeled_compound')
    & ~included_cfm_compounds['inchikey_2d'].isin(development_keys),
    ['inchikey_2d', 'cv_group', 'cv_fold', *LABEL_COLUMNS],
].copy()
synthetic_labels['compound_data_role'] = 'synthetic_only_new_labeled_compound'
compound_labels = pd.concat([dev_labels, synthetic_labels], ignore_index=True)
assert not compound_labels.duplicated('inchikey_2d').any()
compound_labels['n_available_primary_tasks'] = compound_labels[LABEL_COLUMNS].notna().sum(axis=1)
compound_labels['has_real_development_spectrum'] = compound_labels['inchikey_2d'].isin(set(real_dev_index['inchikey_2d']))
compound_labels['has_valid_cfm_spectrum'] = compound_labels['inchikey_2d'].isin(set(cfm_index['inchikey_2d']))
development_cfm_only = (
    compound_labels['compound_data_role'].eq('development')
    & ~compound_labels['has_real_development_spectrum']
    & compound_labels['has_valid_cfm_spectrum']
)
compound_labels.loc[development_cfm_only, 'compound_data_role'] = 'development_without_real_spectrum_cfm_only'
no_selected_target = compound_labels['n_available_primary_tasks'].eq(0)
compound_labels.loc[
    no_selected_target & compound_labels['compound_data_role'].eq('development'),
    'compound_data_role',
] = 'development_no_selected_primary_target'
primary_label_keys = set(compound_labels.loc[compound_labels['n_available_primary_tasks'].gt(0), 'inchikey_2d'])
real_dev_index['has_any_primary_label'] = real_dev_index['inchikey_2d'].isin(primary_label_keys)
atomic_csv(
    real_dev_index,
    OUTPUT_DIR / 'real_development_spectrum_index.csv.gz',
    compression='gzip',
)
atomic_csv(compound_labels, OUTPUT_DIR / 'transformer_compound_labels.csv.gz', compression='gzip')

label_coverage = []
for role, group in compound_labels.groupby('compound_data_role'):
    for task, column in TASKS.items():
        values = pd.to_numeric(group[column], errors='coerce')
        label_coverage.append({
            'compound_data_role': role, 'task': task,
            'n_compounds': len(group), 'n_labeled': int(values.notna().sum()),
            'positive_n': int((values == 1).sum()) if task == 'P_screen_not_ready_binary' else np.nan,
            'negative_n': int((values == 0).sum()) if task == 'P_screen_not_ready_binary' else np.nan,
        })
label_coverage = pd.DataFrame(label_coverage)
atomic_csv(label_coverage, OUTPUT_DIR / 'transformer_label_coverage.csv')
display(label_coverage)

## 8. 外折、内部验证折和CFM双重权限

内部验证折固定为 `(outer_fold + 1) mod 5`。CFM只有同时获得外层测试折和内部验证折两张许可，
才能进入该外折的训练；这样17/18号无需临时重新判断相似性。

In [ ]:
real_dev_index['cv_fold'] = pd.to_numeric(real_dev_index['cv_fold'], errors='raise').astype(int)
permission_base = cfm_permissions.copy()
permission_base['outer_fold'] = permission_base['outer_fold'].astype(int)
permission_base['allowed_in_outer_fold_training'] = as_bool(permission_base['allowed_in_outer_fold_training'])

cfm_fold_frames = []
real_fold_frames = []
for outer_fold in range(N_OUTER_FOLDS):
    inner_fold = (outer_fold + 1) % N_OUTER_FOLDS
    real_part = real_dev_index[['cache_row', 'sample_id', 'inchikey_2d', 'cv_fold', 'has_any_primary_label']].copy()
    real_part['outer_fold'] = outer_fold
    real_part['inner_validation_fold'] = inner_fold
    real_part['split_role'] = np.select(
        [real_part['cv_fold'].eq(outer_fold), real_part['cv_fold'].eq(inner_fold)],
        ['outer_test', 'inner_validation'], default='outer_train',
    )
    real_fold_frames.append(real_part)

    outer_permission = permission_base.loc[
        permission_base['outer_fold'].eq(outer_fold),
        ['inchikey_2d', 'allowed_in_outer_fold_training', 'max_tanimoto_to_outer_test_fold', 'permission_reason'],
    ].rename(columns={
        'allowed_in_outer_fold_training': 'allowed_vs_outer_test',
        'max_tanimoto_to_outer_test_fold': 'max_tanimoto_to_outer_test',
        'permission_reason': 'outer_permission_reason',
    })
    inner_permission = permission_base.loc[
        permission_base['outer_fold'].eq(inner_fold),
        ['inchikey_2d', 'allowed_in_outer_fold_training', 'max_tanimoto_to_outer_test_fold', 'permission_reason'],
    ].rename(columns={
        'allowed_in_outer_fold_training': 'allowed_vs_inner_validation',
        'max_tanimoto_to_outer_test_fold': 'max_tanimoto_to_inner_validation',
        'permission_reason': 'inner_permission_reason',
    })
    cfm_part = cfm_index.merge(outer_permission, on='inchikey_2d', how='left', validate='many_to_one')
    cfm_part = cfm_part.merge(inner_permission, on='inchikey_2d', how='left', validate='many_to_one')
    assert cfm_part[['allowed_vs_outer_test', 'allowed_vs_inner_validation']].notna().all().all()
    cfm_part['outer_fold'] = outer_fold
    cfm_part['inner_validation_fold'] = inner_fold
    cfm_part['allowed_in_training'] = (
        as_bool(cfm_part['allowed_vs_outer_test']) & as_bool(cfm_part['allowed_vs_inner_validation'])
    )
    cfm_part['allowed_primary_pretraining'] = (
        cfm_part['allowed_in_training'] & as_bool(cfm_part['primary_pretraining_eligible'])
    )
    cfm_part['allowed_primary_supervised'] = (
        cfm_part['allowed_in_training'] & as_bool(cfm_part['primary_supervised_eligible'])
    )
    cfm_fold_frames.append(cfm_part)

real_fold_roles = pd.concat(real_fold_frames, ignore_index=True)
cfm_fold_permissions = pd.concat(cfm_fold_frames, ignore_index=True)
atomic_csv(real_fold_roles, OUTPUT_DIR / 'outer_fold_real_roles.csv.gz', compression='gzip')
atomic_csv(cfm_fold_permissions, OUTPUT_DIR / 'outer_fold_cfm_permissions.csv.gz', compression='gzip')

same_cfm = cfm_fold_permissions['generation_role'].eq('same_compound_condition_augmentation')
same_fold = pd.to_numeric(cfm_fold_permissions['cv_fold'], errors='coerce')
forbidden_same = same_cfm & cfm_fold_permissions['allowed_in_training'] & (
    same_fold.eq(cfm_fold_permissions['outer_fold']) |
    same_fold.eq(cfm_fold_permissions['inner_validation_fold'])
)
assert not forbidden_same.any()
print('Real fold-role rows:', len(real_fold_roles))
print('CFM fold-permission rows:', len(cfm_fold_permissions))

## 9. A–F消融路线与逐外折索引

- A：实测谱从头训练。
- B：背景实测谱预训练，随后仅实测谱微调。
- C：实测谱＋同化合物CFM低权重监督增强。
- D：实测谱＋synthetic-only新标签化合物CFM低权重监督增强。
- E：背景实测＋允许的CFM混合预训练，随后仅实测谱微调（主CFM候选）。
- F：CFM-only监督训练、真实谱测试（负面对照）。

In [ ]:
ROUTES = {
    'A_real_scratch': {
        'pretraining': 'none', 'finetuning': 'real_outer_train',
        'purpose': 'deep architecture baseline',
    },
    'B_real_pretrain': {
        'pretraining': 'background_real', 'finetuning': 'real_outer_train',
        'purpose': 'benefit of unlabeled measured-spectrum pretraining',
    },
    'C_same_compound_cfm': {
        'pretraining': 'none', 'finetuning': 'real_outer_train + allowed same-compound CFM 10/20 eV',
        'purpose': 'condition augmentation',
    },
    'D_synthetic_new_cfm': {
        'pretraining': 'none', 'finetuning': 'real_outer_train + allowed synthetic-only CFM 10/20 eV',
        'purpose': 'labeled chemical-space expansion',
    },
    'E_cfm_pretrain_real_finetune': {
        'pretraining': 'background_real + all allowed non-sparse CFM 10/20/40 eV',
        'finetuning': 'real_outer_train only',
        'purpose': 'primary CFM route after sim-to-real audit',
    },
    'F_cfm_only_negative_control': {
        'pretraining': 'none', 'finetuning': 'allowed non-sparse CFM 10/20 eV only',
        'purpose': 'negative control for sim-to-real transfer',
    },
}
route_table = pd.DataFrame([
    {'route': route, **definition} for route, definition in ROUTES.items()
])
atomic_csv(route_table, OUTPUT_DIR / 'transformer_ablation_routes.csv')
atomic_json({
    'routes': ROUTES,
    'inner_validation_rule': '(outer_fold + 1) mod 5',
    'primary_cfm_supervised_weights': CFM_SUPERVISED_WEIGHTS,
    'compound_equal_sampling_required': True,
    'model_or_epoch_selection_may_not_use_outer_test': True,
    'chemvista_may_not_be_read': True,
}, OUTPUT_DIR / 'transformer_ablation_routes.json')

reference_columns = ['sample_id', 'cache_name', 'cache_row', 'inchikey_2d', 'source_pool', 'sample_weight']

def real_refs(frame, sample_weight=1.0):
    out = frame[['sample_id', 'cache_row', 'inchikey_2d', 'source_pool']].copy()
    out['cache_name'] = np.where(
        out['source_pool'].eq('background_unlabeled_real'),
        'real_background_tokens.npz', 'real_development_tokens.npz',
    )
    out['sample_weight'] = float(sample_weight)
    return out[reference_columns]

def cfm_refs(frame, use_primary_weight=True):
    out = frame[['sample_id', 'cache_row', 'inchikey_2d', 'source_pool']].copy()
    out['cache_name'] = 'cfm_tokens.npz'
    if use_primary_weight:
        weights = frame['primary_supervised_sample_weight'].to_numpy(float)
    else:
        weights = np.ones(len(frame), dtype=float)
    out['sample_weight'] = weights
    return out[reference_columns]

index_count_rows = []
written_index_paths = []
for outer_fold in range(N_OUTER_FOLDS):
    fold_path = FOLD_DIR / f'outer_{outer_fold}'
    fold_path.mkdir(parents=True, exist_ok=True)
    real_roles = real_fold_roles.loc[real_fold_roles['outer_fold'].eq(outer_fold)]
    train_ids = set(real_roles.loc[real_roles['split_role'].eq('outer_train'), 'sample_id'])
    validation_ids = set(real_roles.loc[real_roles['split_role'].eq('inner_validation'), 'sample_id'])
    test_ids = set(real_roles.loc[real_roles['split_role'].eq('outer_test'), 'sample_id'])
    labeled_real = as_bool(real_dev_index['has_any_primary_label'])
    real_train = real_dev_index.loc[real_dev_index['sample_id'].isin(train_ids) & labeled_real]
    real_validation = real_dev_index.loc[real_dev_index['sample_id'].isin(validation_ids) & labeled_real]
    real_test = real_dev_index.loc[real_dev_index['sample_id'].isin(test_ids) & labeled_real]

    evaluation = pd.concat([
        real_refs(real_validation).assign(split_role='inner_validation'),
        real_refs(real_test).assign(split_role='outer_test'),
    ], ignore_index=True)
    evaluation_path = fold_path / 'real_evaluation_index.csv.gz'
    atomic_csv(evaluation, evaluation_path, compression='gzip')
    written_index_paths.append(evaluation_path)
    for role, group in evaluation.groupby('split_role'):
        index_count_rows.append({
            'outer_fold': outer_fold, 'route': 'ALL', 'stage': role, 'source_pool': 'development_labeled_real',
            'n_spectra': len(group), 'n_compounds': group['inchikey_2d'].nunique(),
        })

    fold_cfm = cfm_fold_permissions.loc[cfm_fold_permissions['outer_fold'].eq(outer_fold)]
    cfm_pretrain = fold_cfm.loc[as_bool(fold_cfm['allowed_primary_pretraining'])]
    cfm_supervised = fold_cfm.loc[as_bool(fold_cfm['allowed_primary_supervised'])]
    same_supervised = cfm_supervised.loc[
        cfm_supervised['generation_role'].eq('same_compound_condition_augmentation')
    ]
    novel_supervised = cfm_supervised.loc[
        cfm_supervised['generation_role'].eq('synthetic_only_new_labeled_compound')
    ]

    for route in ROUTES:
        if route == 'B_real_pretrain':
            pretrain = real_refs(real_bg_index)
        elif route == 'E_cfm_pretrain_real_finetune':
            pretrain = pd.concat([
                real_refs(real_bg_index), cfm_refs(cfm_pretrain, use_primary_weight=False),
            ], ignore_index=True)
        else:
            pretrain = pd.DataFrame(columns=reference_columns)

        if route in {'A_real_scratch', 'B_real_pretrain', 'E_cfm_pretrain_real_finetune'}:
            finetune = real_refs(real_train)
        elif route == 'C_same_compound_cfm':
            finetune = pd.concat([real_refs(real_train), cfm_refs(same_supervised)], ignore_index=True)
        elif route == 'D_synthetic_new_cfm':
            finetune = pd.concat([real_refs(real_train), cfm_refs(novel_supervised)], ignore_index=True)
        elif route == 'F_cfm_only_negative_control':
            finetune = cfm_refs(cfm_supervised, use_primary_weight=False)
        else:
            raise KeyError(route)

        pretrain_path = fold_path / f'{route}__pretraining.csv.gz'
        finetune_path = fold_path / f'{route}__finetuning.csv.gz'
        atomic_csv(pretrain, pretrain_path, compression='gzip')
        atomic_csv(finetune, finetune_path, compression='gzip')
        written_index_paths.extend([pretrain_path, finetune_path])
        for stage, frame in [('pretraining', pretrain), ('finetuning', finetune)]:
            if len(frame):
                for pool, group in frame.groupby('source_pool'):
                    index_count_rows.append({
                        'outer_fold': outer_fold, 'route': route, 'stage': stage, 'source_pool': pool,
                        'n_spectra': len(group), 'n_compounds': group['inchikey_2d'].nunique(),
                    })
            else:
                index_count_rows.append({
                    'outer_fold': outer_fold, 'route': route, 'stage': stage, 'source_pool': 'none',
                    'n_spectra': 0, 'n_compounds': 0,
                })

fold_index_counts = pd.DataFrame(index_count_rows)
atomic_csv(fold_index_counts, OUTPUT_DIR / 'outer_fold_route_counts.csv')
display(fold_index_counts)

## 10. 缓存完整性、泄漏QC与数据分布图

In [ ]:
def validate_cache(cache_path, index_frame):
    with np.load(cache_path) as cache:
        tokens = cache['peak_tokens']
        padding = cache['padding_mask']
        metadata = cache['metadata_numeric']
        return {
            'rows': int(tokens.shape[0]),
            'shape_ok': tokens.shape == (len(index_frame), MAX_PEAKS, len(PEAK_TOKEN_COLUMNS)),
            'padding_shape_ok': padding.shape == (len(index_frame), MAX_PEAKS),
            'metadata_shape_ok': metadata.shape == (len(index_frame), len(META_NUMERIC_COLUMNS)),
            'finite_tokens': bool(np.isfinite(tokens).all()),
            'finite_metadata': bool(np.isfinite(metadata).all()),
            'nonempty_rows': int((~padding).sum(axis=1).astype(bool).sum()),
        }

cache_qc = {
    'real_development': validate_cache(CACHE_DIR / 'real_development_tokens.npz', real_dev_index),
    'real_background': validate_cache(CACHE_DIR / 'real_background_tokens.npz', real_bg_index),
    'cfm': validate_cache(CACHE_DIR / 'cfm_tokens.npz', cfm_index),
}
atomic_json(cache_qc, OUTPUT_DIR / 'transformer_token_cache_qc.json')

cfm_train_allowed = cfm_fold_permissions.loc[as_bool(cfm_fold_permissions['allowed_in_training'])]
same_allowed = cfm_train_allowed.loc[
    cfm_train_allowed['generation_role'].eq('same_compound_condition_augmentation')
]
same_fold_numeric = pd.to_numeric(same_allowed['cv_fold'], errors='coerce')
no_same_identity_fold_leak = bool(
    (~same_fold_numeric.eq(same_allowed['outer_fold'])).all()
    and (~same_fold_numeric.eq(same_allowed['inner_validation_fold'])).all()
)
synthetic_allowed = cfm_train_allowed.loc[
    cfm_train_allowed['generation_role'].eq('synthetic_only_new_labeled_compound')
    & cfm_train_allowed['supervised_compound_role'].eq('property_without_public_spectrum')
]
synthetic_similarity_ok = bool(
    (pd.to_numeric(synthetic_allowed['max_tanimoto_to_outer_test'], errors='coerce') < 0.40).all()
    and (pd.to_numeric(synthetic_allowed['max_tanimoto_to_inner_validation'], errors='coerce') < 0.40).all()
)

all_index_cache_names = set()
all_finetune_synthetic_ids = set()
for path in written_index_paths:
    frame = pd.read_csv(path)
    if len(frame):
        all_index_cache_names |= set(frame['cache_name'].astype(str))
        if path.name.endswith('__finetuning.csv.gz'):
            all_finetune_synthetic_ids |= set(frame.loc[frame['source_pool'].eq('cfm'), 'sample_id'].astype(str))

qc_rows = [
    {'check': 'cfm15_ready', 'passed': cfm15['status'] == 'READY_FOR_NOTEBOOK_16', 'detail': cfm15['status']},
    {'check': 'cfm_usage_rule_locked', 'passed': cfm15['cfm_usage_recommendation'] == 'LOW_WEIGHT_AUGMENTATION_WITH_DOMAIN_LABEL', 'detail': cfm15['cfm_usage_recommendation']},
    {'check': 'chemvista_remains_locked', 'passed': cfm15['external_test_unlocked'] is False, 'detail': 'no ChemVista spectrum path in PATHS'},
    {'check': 'split_and_spectrum_qc_pass', 'passed': split_qc['status'].eq('PASS').all() and ~spectrum_qc['status'].eq('FAIL').any(), 'detail': split_freeze['split_version']},
    {'check': 'real_development_cache_complete', 'passed': cache_qc['real_development']['rows'] == len(real_dev_index) and cache_qc['real_development']['shape_ok'], 'detail': len(real_dev_index)},
    {'check': 'real_background_cache_complete', 'passed': cache_qc['real_background']['rows'] == len(real_bg_index) and cache_qc['real_background']['shape_ok'], 'detail': len(real_bg_index)},
    {'check': 'cfm_cache_complete', 'passed': cache_qc['cfm']['rows'] == len(cfm_index) and cache_qc['cfm']['shape_ok'], 'detail': len(cfm_index)},
    {'check': 'all_cache_values_finite', 'passed': all(item['finite_tokens'] and item['finite_metadata'] for item in cache_qc.values()), 'detail': str(cache_qc)},
    {'check': 'background_disjoint_development_external_cfm', 'passed': set(real_bg_index['inchikey_2d']).isdisjoint(development_keys | external_keys | set(cfm_index['inchikey_2d'])), 'detail': real_bg_index['inchikey_2d'].nunique()},
    {'check': 'development_real_disjoint_novel', 'passed': set(real_dev_index['inchikey_2d']).isdisjoint(novel_keys), 'detail': len(set(real_dev_index['inchikey_2d']) & novel_keys)},
    {'check': 'cfm_disjoint_chemvista_novel_identities', 'passed': set(cfm_index['inchikey_2d']).isdisjoint(novel_keys), 'detail': len(set(cfm_index['inchikey_2d']) & novel_keys)},
    {'check': 'same_compound_cfm_blocked_from_test_and_validation', 'passed': no_same_identity_fold_leak, 'detail': len(same_allowed)},
    {'check': 'synthetic_only_below_similarity_threshold_for_test_and_validation', 'passed': synthetic_similarity_ok, 'detail': len(synthetic_allowed)},
    {'check': 'evaluation_indices_real_only', 'passed': all(path.name != 'real_evaluation_index.csv.gz' or set(pd.read_csv(path)['source_pool']) <= {'development_labeled_real'} for path in written_index_paths), 'detail': 'all outer folds'},
    {'check': 'only_known_caches_referenced', 'passed': all_index_cache_names <= {'real_development_tokens.npz', 'real_background_tokens.npz', 'cfm_tokens.npz'}, 'detail': str(sorted(all_index_cache_names))},
    {'check': 'supervised_cfm_is_primary_eligible', 'passed': all_finetune_synthetic_ids <= set(cfm_index.loc[as_bool(cfm_index['primary_supervised_eligible']), 'sample_id'].astype(str)), 'detail': len(all_finetune_synthetic_ids)},
    {'check': 'all_six_routes_present_each_fold', 'passed': fold_index_counts.loc[fold_index_counts['route'].ne('ALL')].groupby('outer_fold')['route'].nunique().eq(len(ROUTES)).all(), 'detail': len(ROUTES)},
]
qc = pd.DataFrame(qc_rows)
atomic_csv(qc, OUTPUT_DIR / 'transformer_data_freeze_qc.csv')
display(qc)

cfm_plot = cfm_index.groupby(['mode', 'collision_energy_value']).agg(
    all_spectra=('sample_id', 'size'),
    primary_pretraining=('primary_pretraining_eligible', 'sum'),
    primary_supervised=('primary_supervised_eligible', 'sum'),
).reset_index()
plot_long = cfm_plot.melt(
    id_vars=['mode', 'collision_energy_value'], var_name='eligibility', value_name='spectra'
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_long, x='collision_energy_value', y='spectra', hue='eligibility', ax=axes[0])
axes[0].set_title('CFM spectra retained for Transformer routes')
route_plot = fold_index_counts.loc[
    fold_index_counts['stage'].eq('finetuning') & fold_index_counts['route'].ne('ALL')
].groupby(['route', 'source_pool'], as_index=False)['n_spectra'].mean()
sns.barplot(data=route_plot, x='route', y='n_spectra', hue='source_pool', ax=axes[1])
axes[1].tick_params(axis='x', rotation=35)
axes[1].set_title('Mean fine-tuning spectra per outer fold')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '16_transformer_data_freeze_summary.png', dpi=240, bbox_inches='tight')
plt.show()

## 11. 最终冻结清单

正式运行要求全部QC通过。DRY_RUN只用于代码验证，不能进入17号。

In [ ]:
all_qc_pass = bool(as_bool(qc['passed']).all())
if DRY_RUN:
    FREEZE_STATUS = 'DRY_RUN_VALIDATION_ONLY'
elif all_qc_pass:
    FREEZE_STATUS = 'READY_FOR_NOTEBOOK_17'
else:
    FREEZE_STATUS = 'BLOCKED_REVIEW_QC'

formal_outputs = [
    OUTPUT_DIR / 'transformer_categorical_vocab.json',
    OUTPUT_DIR / 'transformer_token_schema.json',
    OUTPUT_DIR / 'real_development_spectrum_index.csv.gz',
    OUTPUT_DIR / 'real_background_pretraining_index.csv.gz',
    OUTPUT_DIR / 'cfm_transformer_spectrum_index.csv.gz',
    OUTPUT_DIR / 'transformer_compound_labels.csv.gz',
    OUTPUT_DIR / 'transformer_label_coverage.csv',
    OUTPUT_DIR / 'outer_fold_real_roles.csv.gz',
    OUTPUT_DIR / 'outer_fold_cfm_permissions.csv.gz',
    OUTPUT_DIR / 'transformer_ablation_routes.csv',
    OUTPUT_DIR / 'transformer_ablation_routes.json',
    OUTPUT_DIR / 'outer_fold_route_counts.csv',
    OUTPUT_DIR / 'transformer_token_cache_qc.json',
    OUTPUT_DIR / 'transformer_data_freeze_qc.csv',
    CACHE_DIR / 'real_development_tokens.npz',
    CACHE_DIR / 'real_background_tokens.npz',
    CACHE_DIR / 'cfm_tokens.npz',
    FIGURE_DIR / '16_transformer_data_freeze_summary.png',
    *written_index_paths,
]
formal_outputs = sorted(set(path for path in formal_outputs if path.exists()), key=lambda path: str(path))

manifest = {
    'notebook': '16_Transformer预训练与微调数据冻结.ipynb',
    'generated_at_utc': utc_now(),
    'status': FREEZE_STATUS,
    'external_test_unlocked': False,
    'chemvista_spectra_used': False,
    'scientific_scope': 'Fold-isolated real/CFM Transformer data freeze; no model training and no external evaluation',
    'upstream': {
        'cfm15_status': cfm15['status'],
        'cfm15_usage_recommendation': cfm15['cfm_usage_recommendation'],
        'split_version': split_freeze['split_version'],
        'spectrum_audit_version': spectrum_audit['audit_version'],
        'public_stream_sha256_from_frozen_audit': spectrum_audit['public_stream']['sha256'],
    },
    'token_schema': {
        'max_peaks': MAX_PEAKS, 'peak_features': PEAK_TOKEN_COLUMNS,
        'metadata_numeric': META_NUMERIC_COLUMNS,
        'categorical_vocab_file': 'transformer_categorical_vocab.json',
    },
    'cfm_policy': {
        'minimum_primary_peaks': MIN_PRIMARY_CFM_PEAKS,
        'primary_pretraining_energies_ev': sorted(PRIMARY_CFM_PRETRAIN_ENERGIES),
        'primary_supervised_energies_ev': sorted(PRIMARY_CFM_SUPERVISED_ENERGIES),
        'primary_supervised_weights': CFM_SUPERVISED_WEIGHTS,
        '40eV_policy': 'pretraining only',
        'sparse_policy': 'retained in cache and sensitivity pool; excluded from primary routes',
        'domain_label_required': True,
    },
    'fold_policy': {
        'n_outer_folds': N_OUTER_FOLDS,
        'inner_validation_rule': '(outer_fold + 1) mod 5',
        'evaluation_domain': 'real Development spectra only',
        'cfm_permission': 'must pass both outer-test and inner-validation identity/similarity blocks',
    },
    'routes': ROUTES,
    'counts': {
        'real_development_spectra': len(real_dev_index),
        'real_development_compounds': int(real_dev_index['inchikey_2d'].nunique()),
        'real_background_spectra': len(real_bg_index),
        'real_background_compounds': int(real_bg_index['inchikey_2d'].nunique()),
        'cfm_cached_spectra': len(cfm_index),
        'cfm_cached_compounds': int(cfm_index['inchikey_2d'].nunique()),
        'cfm_primary_pretraining_spectra': int(as_bool(cfm_index['primary_pretraining_eligible']).sum()),
        'cfm_primary_supervised_spectra': int(as_bool(cfm_index['primary_supervised_eligible']).sum()),
        'compound_label_rows': len(compound_labels),
        'fold_index_files': len(written_index_paths),
    },
    'software': {
        'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__,
        'platform': platform.platform(),
    },
    'input_hashes': {
        name: sha256_file(path) for name, path in PATHS.items()
        if name not in {'public_stream'}
    },
    'output_hashes': {
        str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in formal_outputs
    },
}
atomic_json(manifest, OUTPUT_DIR / 'transformer_data_freeze_manifest.json')
print('FREEZE_STATUS:', FREEZE_STATUS)
print('Manifest:', OUTPUT_DIR / 'transformer_data_freeze_manifest.json')
if FREEZE_STATUS == 'READY_FOR_NOTEBOOK_17':
    print('16号完成，可以进入17号Transformer自监督预训练。')
elif FREEZE_STATUS == 'DRY_RUN_VALIDATION_ONLY':
    print('这只是DRY_RUN；改回DRY_RUN=False后完整运行。')
else:
    print('存在未通过QC，不得进入17号。')

## 下一步

17号只完成外折独立的自监督预训练，并保存每折检查点：

- B路线：公共背景实测谱预训练；
- E路线：公共背景实测谱＋该外折允许的CFM混合预训练；
- masked peak reconstruction；
- 同结构实测谱对比学习；
- 训练侧real–synthetic一致性；
- 不读取外折测试谱标签，也不读取ChemVista。

18号才执行P/B/T微调、A–F消融、3个随机种子和化合物级配对统计。